# Pemodelan Rantai Markov untuk Pergerakan IHSG

Penelitian ini memodelkan perubahan Indeks Harga Saham Gabungan (IHSG) sebagai rantai Markov diskrit dengan tiga keadaan: Turun, Stabil, dan Naik. Keadaan ditentukan berdasarkan persentase *return* harian.

Model ini mengestimasi probabilitas transisi menggunakan data historis dengan asumsi bahwa keadaan saat ini telah merepresentasikan seluruh informasi sebelumnya (sifat Markov). Hasil estimasi digunakan untuk menghitung probabilitas pergerakan IHSG pada dua dan lima hari perdagangan berikutnya.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Konfigurasi visualisasi
sns.set_theme(style="whitegrid")

## 1. Memuat Data dan Mendefinisikan Ruang Keadaan ($S$)

Data IHSG dimuat dari tahap *preprocessing*. Karena IHSG adalah data kontinu yang tidak bisa dimodelkan dengan rantai Markov berkeadaan diskrit, pergerakannya diubah menjadi persentase return harian menggunakan rumus:

$$ R_t = \frac{I_t - I_{t-1}}{I_{t-1}} \times 100\% $$

In [ ]:
# Memuat data
df = pd.read_csv(r"E:\University\Stokastik\Tugas 2\code\data\raw\ihsg_daily.csv")

# Pastikan kolom Date bertipe datetime dan urutkan waktu
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# Menghitung persentase return harian dari kolom 'Adj Close'
df['Return_Pct'] = df['Adj Close'].pct_change() * 100

# Menghapus baris pertama (karena tidak memiliki nilai return)
df = df.dropna(subset=['Return_Pct']).reset_index(drop=True)

# Tampilkan 5 data teratas
display(df[['Date', 'Adj Close', 'Return_Pct']].head())

### Menentukan Batas Keadaan

Ambang batas $\pm 0.5\%$ digunakan untuk membagi *return* IHSG ke dalam tiga keadaan diskrit. Ruang keadaan ($S$) didefinisikan sebagai $S = \{\text{Turun}, \text{Stabil}, \text{Naik}\}$ dengan aturan:

$$ X_t = \begin{cases} 
\text{Turun}, & R_t < -0.5\% \\ 
\text{Stabil}, & -0.5\% \le R_t \le 0.5\% \\ 
\text{Naik}, & R_t > 0.5\% 
\end{cases} $$

In [ ]:
# Fungsi untuk menentukan state berdasarkan return
def get_state(r):
    if r < -0.5:
        return 'Turun'
    elif r > 0.5:
        return 'Naik'
    else:
        return 'Stabil'

# Mengaplikasikan fungsi pada data
df['State'] = df['Return_Pct'].apply(get_state)

# Melihat distribusi jumlah hari untuk masing-masing state
state_counts = df['State'].value_counts()
display(state_counts)

# Visualisasi distribusi State
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='State', order=['Turun', 'Stabil', 'Naik'], hue='State', palette='pastel', legend=False)
plt.title('Distribusi Keadaan Harian IHSG')
plt.xlabel('Keadaan')
plt.ylabel('Frekuensi (Hari)')
plt.show()

## 2. Menyusun Matriks Probabilitas Transisi ($P$)

Probabilitas transisi $P$ diestimasi dari frekuensi historis. Jika $N_{ij}$ adalah jumlah transisi dari keadaan $i$ ke keadaan $j$, dan $N_i$ adalah total dari keadaan $i$, maka peluang transisinya adalah:

$$ P_{ij} = \frac{N_{ij}}{N_i} $$

Satu langkah transisi mewakili satu hari perdagangan bursa (hari libur diabaikan). Model juga mengasumsikan bahwa matriks $P$ bersifat stasioner.

In [ ]:
# Membuat pasangan keadaan: State Hari Ini (t) vs State Besok (t+1)
df['Next_State'] = df['State'].shift(-1)

# Menghapus baris terakhir karena tidak memiliki hari esok
df_transitions = df.dropna(subset=['Next_State'])

# Menyusun Crosstab (Frekuensi Transisi N_ij)
state_order = ['Turun', 'Stabil', 'Naik']
transition_freq = pd.crosstab(df_transitions['State'], df_transitions['Next_State'])

# Memastikan urutan baris dan kolom sesuai (T, S, N)
transition_freq = transition_freq.reindex(index=state_order, columns=state_order, fill_value=0)

# Menampilkan Matriks Frekuensi
print("Matriks Frekuensi Transisi (N_ij):")
display(transition_freq)

# Normalisasi baris (Membagi tiap elemen baris dengan total baris N_i)
transition_prob = transition_freq.div(transition_freq.sum(axis=1), axis=0)

print("\nMatriks Probabilitas Transisi (P):")
display(transition_prob)

# Visualisasi Heatmap Matriks P
plt.figure(figsize=(6, 5))
sns.heatmap(transition_prob, annot=True, fmt=".4f", cmap="Blues", cbar=True)
plt.title("Heatmap Matriks Probabilitas Transisi P")
plt.xlabel("Keadaan Besok (t+1)")
plt.ylabel("Keadaan Hari Ini (t)")
plt.show()

### Syarat Matriks Stokastik

Sebuah matriks probabilitas transisi dikatakan valid jika memenuhi dua syarat:
1. Semua probabilitas bernilai non-negatif ($P_{ij} \ge 0$).
2. Jumlah probabilitas pada setiap baris sama dengan 1 ($\sum_j P_{ij} = 1$).

In [ ]:
# 1. Cek non-negatif
is_non_negative = (transition_prob >= 0).all().all()

# 2. Cek jumlahan baris sama dengan 1
# (Menggunakan np.isclose untuk menghindari error pembulatan floating point)
row_sums = transition_prob.sum(axis=1)
is_row_sum_one = np.isclose(row_sums, 1).all()

print(f"Semua elemen non-negatif? {is_non_negative}")
print("Jumlah probabilitas tiap baris:")
print(row_sums)
print(f"\nKesimpulan: Matriks P {'VALID' if (is_non_negative and is_row_sum_one) else 'TIDAK VALID'} sebagai matriks stokastik.")

## 3. Menghitung Probabilitas Transisi n-Langkah ($P^2$ dan $P^5$)

Perkalian matriks digunakan untuk menghitung probabilitas pergerakan IHSG pada hari perdagangan ke-$n$.
- $P^2$: Probabilitas IHSG pada dua hari perdagangan berikutnya.
- $P^5$: Probabilitas IHSG pada lima hari perdagangan berikutnya.

In [ ]:
P = transition_prob.values

# Menghitung P^2
P_2 = np.linalg.matrix_power(P, 2)
df_P2 = pd.DataFrame(P_2, index=state_order, columns=state_order)

print("Matriks Probabilitas Transisi 2 Hari (P^2):")
display(df_P2)

# Menghitung P^5
P_5 = np.linalg.matrix_power(P, 5)
df_P5 = pd.DataFrame(P_5, index=state_order, columns=state_order)

print("\nMatriks Probabilitas Transisi 5 Hari (P^5):")
display(df_P5)

## 4. Interpretasi Praktis (Untuk Makalah)

Cara membaca matriks probabilitas transisi $P^5$, contohnya pada baris terakhir (Naik):
> "Jika IHSG hari ini Naik, maka peluang arah IHSG pada lima hari perdagangan ke depan adalah $X\%$ Turun, $Y\%$ Stabil, dan $Z\%$ Naik." *(Ganti angka sesuai tabel).*

Model ini berguna bagi *trader* dan analis risiko untuk memetakan peluang momentum arah pasar secara persentase, bukan untuk memprediksi harga IHSG. Hasil probabilitas dari matriks $P$ hingga $P^n$ dapat menunjukkan apakah tren pasar lebih kuat untuk bertahan pada arah yang sama, atau lebih cenderung berbalik arah (*mean-reverting*) dalam jangka waktu tertentu.